In [0]:
# COMMAND ----------
# MAGIC %md
# MAGIC # Landing to Bronze
# MAGIC Notebook responsável pela ingestão de dados brutos (CSVs) e da API do Banco Central (Cotação do Dólar) para a camada Bronze.

# COMMAND ----------
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp
from datetime import datetime, timedelta

# COMMAND ----------
# Criação do banco de dados da camada Bronze
spark.sql("CREATE DATABASE IF NOT EXISTS bronze;")

# COMMAND ----------
# Caminho do Volume onde os 5 arquivos CSV foram carregados
# ATENÇÃO: Substitua pelo caminho correto do seu Volume no Databricks
volume_path = "/Volumes/workspace/default/inputs/" 

# Mapeamento de arquivos para as respectivas tabelas Bronze
files_mapping = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews"
}

# Processamento dos arquivos CSV
for file_name, table_name in files_mapping.items():
    file_path = f"{volume_path}{file_name}"
    
    try:
        # Leitura do CSV (sem qualquer alteração estrutural ou de conteúdo)
        df = spark.read.csv(file_path, header=True, inferSchema=True)
        
        # Adição da coluna com o timestamp exato do momento da inserção
        df_bronze = df.withColumn("ingestion_datetime", current_timestamp())
        
        # Gravação das tabelas em formato Delta, utilizando o modo Append
        df_bronze.write.format("delta").mode("append").saveAsTable(f"bronze.{table_name}")
        print(f"Tabela bronze.{table_name} atualizada com sucesso.")
        
    except Exception as e:
        print(f"Erro ao processar o arquivo {file_name}: {e}")

# COMMAND ----------
# Configuração dos Widgets para a API do Banco Central
# Formato esperado: MM-DD-AAAA. Sugestão: últimos 7 dias corridos.
data_fim_default = datetime.now().strftime("%m-%d-%Y")
data_inicio_default = (datetime.now() - timedelta(days=7)).strftime("%m-%d-%Y")

dbutils.widgets.text("data_inicio", data_inicio_default, "Data Início (MM-DD-YYYY)")
dbutils.widgets.text("data_fim", data_fim_default, "Data Fim (MM-DD-YYYY)")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

# COMMAND ----------
# Ingestão da API do Banco Central (Cotação do Dólar)
url_bcb = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo"
    f"(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    f"&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

response = requests.get(url_bcb)

if response.status_code == 200:
    dados_cotacao = response.json().get("value", [])
    
    if dados_cotacao:
        # Converte a lista de dicionários da API para um Spark DataFrame via Pandas
        pdf_cotacao = pd.DataFrame(dados_cotacao)
        df_bcb = spark.createDataFrame(pdf_cotacao)
        
        # Adiciona a coluna de ingestão
        df_bcb_bronze = df_bcb.withColumn("ingestion_datetime", current_timestamp())
        
        # Salva na tabela tb_cotacao_dolar na camada Bronze (Delta/Append)
        df_bcb_bronze.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
        print("Tabela bronze.tb_cotacao_dolar atualizada com sucesso.")
    else:
        print("Nenhum dado de cotação retornado para o período selecionado (API não retorna em fins de semana/feriados).")
else:
    print(f"Erro na requisição da API BCB: Status Code {response.status_code}")

Tabela bronze.tb_movies_info atualizada com sucesso.
Tabela bronze.tb_movies_financials atualizada com sucesso.
Tabela bronze.tb_movies_metrics atualizada com sucesso.
Tabela bronze.tb_credits_and_tags atualizada com sucesso.
Tabela bronze.tb_movies_reviews atualizada com sucesso.
Tabela bronze.tb_cotacao_dolar atualizada com sucesso.
